In [22]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd

from sklearn.ensemble         import RandomForestClassifier, StackingClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.model_selection  import StratifiedKFold
from sklearn.metrics          import roc_auc_score, recall_score, f1_score
from xgboost                  import XGBClassifier
from lightgbm                 import LGBMClassifier

from utils.preprocessing        import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils           import get_model_train_eval
from utils.feature_engineering  import drop_highly_correlated_features


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def santander_base_job2(columns=['ID'], zcr=0.99, savedDrop=False, 
                       isScaled=False, isSplit=False, 
                       apply_log=False, log_threshold=3.0):  # ✅ 추가
    '''
    SantaderBank CS 분석을 위한 기본 작업
    
    Args:
        columns : list, 1차 삭제할 컬럼
        zcr : float, zero count rate 삭제 기준값
        savedDrop : bool, zcr로 삭제한 컬럼 저장 여부 
        isScaled : bool, scale 여부
        isSplit : bool, train/val split 여부
        apply_log : bool, log1p 변환 적용 여부 (NEW)
        log_threshold : float, log 변환 적용할 skewness 기준값 (NEW)
    
    Returns:   
        isSplit=True  => X_train, X_val, y_train, y_val 
        isSplit=False => X_reduced, y_labels, X_test_reduced
    
    Example:
        # 기본 사용
        X_reduced, y_labels, X_test_reduced = santander_base_job2()
        
        # log 변환 + scaling + split
        X_train, X_val, y_train, y_val = santander_base_job2(
            isScaled=True, 
            isSplit=True, 
            apply_log=True,
            log_threshold=3.0
        )
    '''
    # 데이터 로딩 및 기본 전처리
    train, test = load_data()
    X_features, y_labels = split_features_target(train)
    X_test = test.drop(columns=columns, axis=1)

    # zero_count_rate 제거
    X_features, X_test = remove_zero_columns2(X_features, X_test, zcr)  
    print(f"Zero columns 제거 후: {X_features.shape}")

    # var3 처리
    X_features['var3'] = X_features['var3'].replace(-999999, 2)
    X_test['var3'] = X_test['var3'].replace(-999999, 2)
    
    # 상관계수 높은 feature들 삭제하기
    X_reduced, to_drop = drop_highly_correlated_features(X_features)
    X_test_reduced = X_test.drop(to_drop, axis=1)  
    print(f"상관계수 높은 컬럼 제거 후: {X_reduced.shape}, {X_test_reduced.shape}")
    print(f"삭제된 컬럼 개수: {len(to_drop)}")

    if savedDrop:
        series = pd.Series(sorted(to_drop), name="Dropped_Columns")
        series.to_csv(f"../doc/DroppedColumnsCorr95_{len(to_drop)}.csv", index=False)  
        print(f"✓ 삭제된 컬럼 목록 저장 완료")    
    
    # ========================================
    # ✅ NEW: Log 변환 (스케일링 전에!)
    # ========================================
    if apply_log:
        print(f"\n{'='*60}")
        print(f"Log1p 변환 적용 (skewness > {log_threshold})")
        print(f"{'='*60}")
        
        # Train 데이터 분석
        skewness = X_reduced.skew().sort_values(ascending=False)
        high_skew_cols = skewness[abs(skewness) > log_threshold].index.tolist()
        
        # 음수 값이 있는 컬럼 제외
        valid_log_cols = []
        for col in high_skew_cols:
            if (X_reduced[col] >= 0).all():  # 모든 값이 0 이상
                valid_log_cols.append(col)
            else:
                print(f"  ⚠️  {col}: 음수 값 존재 -> log 변환 제외")
        
        if len(valid_log_cols) > 0:
            print(f"\nLog 변환 적용할 컬럼 ({len(valid_log_cols)}개):")
            for i, col in enumerate(valid_log_cols[:10], 1):
                print(f"  {i:2d}. {col:20s} (skewness: {skewness[col]:>7.2f})")
            if len(valid_log_cols) > 10:
                print(f"       ... 외 {len(valid_log_cols)-10}개")
            
            # Train에 log 변환 적용
            X_reduced[valid_log_cols] = np.log1p(X_reduced[valid_log_cols])
            
            # Test에도 동일하게 적용
            X_test_reduced[valid_log_cols] = np.log1p(X_test_reduced[valid_log_cols])
            
            print(f"\n✓ Log1p 변환 완료: {len(valid_log_cols)}개 컬럼")
            
            series = pd.Series(valid_log_cols, name="Log1p_Colums")
            series.to_csv("../doc/Log1pColumns.txt", index=False)  
            print(f"✓ Log1p 변환 목록 저장 완료")   
            
            # 변환 후 왜도 확인
            new_skewness = X_reduced[valid_log_cols].skew()
            print(f"\n변환 후 평균 왜도: {abs(skewness[valid_log_cols]).mean():.2f} -> {abs(new_skewness).mean():.2f}")
        else:
            print("✓ Log 변환 적용 가능한 컬럼이 없습니다.")
        
        print(f"{'='*60}\n")
        
        
    # ========================================
    # 스케일링 및 Split
    # ========================================
    if isSplit:           
        if isScaled:
            # 스케일링 (log 변환 후)
            X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)        
            # 학습/검증 데이터 분리
            X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)
            print(f"✓ 스케일링 + Split 완료")
        else:
            # 학습/검증 데이터 분리
            X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)
            print(f"✓ Split 완료")
        
        return X_train, X_val, y_train, y_val 
    else:
        return X_reduced, y_labels, X_test_reduced

In [24]:
rf_best_param = {
    "random_state": 23,
    "n_estimators": 390,
    "max_depth": 25,
    "class_weight": {0: 1, 1: 2},
    "min_samples_leaf": 1,
    "min_samples_split": 7,
    "n_jobs": -1
}

In [ ]:
# ========================================
# 1. 기본 사용 (기존과 동일)
# ========================================
# X_reduced, y_labels, X_test_reduced = santander_base_job2()


In [ ]:
# ========================================
# 2. Log 변환 + Scaling + Split
# ========================================
# X_reduced, y_labels, X_test_reduced = santander_base_job2(
#     isScaled=False, 
#     isSplit=False, 
#     savedDrop=True,
#     apply_log=True,
#     log_threshold=3.0  # skewness > 3인 컬럼만 변환
# )



In [ ]:
# ========================================
# 3. 비교 실험
# ========================================

# Case 1: Log 변환 없이
X_train1, X_val1, y_train1, y_val1 = santander_base_job2(
    isScaled=True, 
    isSplit=True, 
    apply_log=False
)
model_name1 = 'RandomForest(99per95corrBestHO)_noLog_20251124'


rf1 = RandomForestClassifier(**rf_best_param)
# 함수 이용
get_model_train_eval(rf1, model_name1, X_train1, X_val1, y_train1, y_val1)
# rf1.fit(X_train1, y_train1)
# pred_proba1 = rf1.predict_proba(X_val1)[:, 1]
# auc1 = roc_auc_score(y_val1, pred_proba1)

# AUC: 0.8406, 정확도: 0.9602, 정밀도: 0.3846, 재현율: 0.0083, F1: 0.0163

In [ ]:
# Case 2: Log 변환 적용
X_train2, X_val2, y_train2, y_val2 = santander_base_job2(
    isScaled=True, 
    isSplit=True, 
    apply_log=True,
    log_threshold=3.0
)
model_name2 = 'RandomForest(99per95corrBestHO)_Log1p_20251124'
rf2 = RandomForestClassifier(**rf_best_param)
get_model_train_eval(rf2, model_name2, X_train2, X_val2, y_train2, y_val2)


In [ ]:
# rf2.fit(X_train2, y_train2)
# pred_proba2 = rf2.predict_proba(X_val2)[:, 1]
# auc2 = roc_auc_score(y_val2, pred_proba2)

# print(f"\n{'='*60}")
# print(f"성능 비교")
# print(f"{'='*60}")
# print(f"Log 변환 없음: AUC = {auc1:.4f}")
# print(f"Log 변환 적용: AUC = {auc2:.4f}")
# print(f"성능 향상: {(auc2-auc1)*100:+.2f}%")
# print(f"{'='*60}")

# rf1 - AUC: 0.8406, 정확도: 0.9602, 정밀도: 0.3846, 재현율: 0.0083, F1: 0.0163
# rf2 - AUC: 0.8406, 정확도: 0.9602, 정밀도: 0.3846, 재현율: 0.0083, F1: 0.0163

✓ 모델 저장 완료: ../models\RandomForest(99per95corrBestHO)_Log1p_20251124.pkl
  파일 크기: 79.13 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8406, 정확도: 0.9602, 정밀도: 0.3846, 재현율: 0.0083, F1: 0.0163
오차행렬:
[[14594     8]
 [  597     5]]
실행 시간: 11.452883243560791


In [ ]:
def analyze_skewness(df, threshold=3.0):
    """
    데이터프레임의 왜도를 분석하고 log 변환 필요 여부 판단
    
    Args:
        df : pd.DataFrame
        threshold : float, 왜도 기준값
    """
    skewness = df.skew().sort_values(ascending=False)
    
    print(f"{'='*80}")
    print(f"왜도(Skewness) 분석")
    print(f"{'='*80}")
    print(f"전체 컬럼 수: {len(df.columns)}")
    print(f"왜도 > {threshold}: {len(skewness[abs(skewness) > threshold])}개")
    print(f"왜도 > 5: {len(skewness[abs(skewness) > 5])}개")
    print(f"왜도 > 10: {len(skewness[abs(skewness) > 10])}개")
    
    print(f"\n상위 20개 컬럼:")
    print(f"{'-'*80}")
    print(f"{'순위':>4} {'컬럼명':20} {'왜도':>10} {'최소값':>12} {'최대값':>12} {'Log변환':>10}")
    print(f"{'-'*80}")
    
    for i, (col, skew_val) in enumerate(skewness.head(20).items(), 1):
        min_val = df[col].min()
        max_val = df[col].max()
        log_ok = "가능" if min_val >= 0 else "불가(음수)"
        
        print(f"{i:4d} {col:20s} {skew_val:10.2f} {min_val:12.2f} {max_val:12.2f} {log_ok:>10}")
    
    print(f"{'='*80}\n")
    
    # 음수 값 분석
    negative_cols = [col for col in df.columns if (df[col] < 0).any()]
    print(f"음수 값이 있는 컬럼: {len(negative_cols)}개")
    if len(negative_cols) > 0:
        print(f"  예시: {negative_cols[:5]}")




In [ ]:
# 사용
train, test = load_data()
X_features, y_labels = split_features_target(train)
analyze_skewness(X_features, threshold=3.0)

In [ ]:
# 먼저 분석해보고
# analyze_skewness(X_features)

# 왜도가 큰 컬럼이 많다면 실험
# X_train1, X_val1, y_train1, y_val1 = santander_base_job2(isScaled=True, isSplit=True, apply_log=False)
# X_train2, X_val2, y_train2, y_val2 = santander_base_job2(isScaled=True, isSplit=True, apply_log=True)

# 성능 비교 후 결정!

In [ ]:
X_reduced, y_labels, X_test_reduced = santander_base_job2(
    isScaled=True, 
    isSplit=False, 
    apply_log=True,
    log_threshold=3.0  # skewness > 3인 컬럼만 변환
)

In [26]:
# 모델 앙상블

# 1. Base models 정의
# 최적 하이퍼파라미터: 
# {'max_depth': np.float64(25.0), 
# 'min_samples_leaf': np.float64(1.0), 
# 'min_samples_split': np.float64(7.0), 
# 'n_estimators': np.float64(390.0)}
rf_clf = RandomForestClassifier(**rf_best_param)

# 최적 하이퍼파라미터: 
# 'n_estimators': np.float64(320.0), 
# 'colsample_bytree': np.float64(0.8828333771389649), 
# 'gamma': np.float64(0.058408742978283044), 
# 'learning_rate': np.float64(0.12721361071578832), 
# 'max_depth': np.float64(6.0), 
# 'min_child_weight': np.float64(2.0), 
# 'subsample': np.float64(0.845580758889997)}
xgb_clf = XGBClassifier(
    random_state      = 23,    
    n_estimators      = 320,
    colsample_bytree  = 0.88,
    gamma             = 0.058, 
    learning_rate     = 0.13,
    max_depth         = 6,
    scale_pos_weight  = 10,
    min_child_weight  = 2, 
    subsample         = 0.85,    
    eval_metric       ='auc',
    use_label_encoder = False,
    n_jobs            = -1,
)

# 최적 하이퍼파라미터: {
# 'n_estimators': np.float64(660.0), 
# 'colsample_bytree': np.float64(0.7342406115797382), 
# 'learning_rate': np.float64(0.02751826185901235), 
# 'num_leaves': np.float64(42.0), 
# 'reg_alpha': np.float64(0.6127977982911577), 
# 'reg_lambda': np.float64(0.1262561992869149), 
# 'subsample': np.float64(0.973776538266153)}
lgbm_best_param = {
    'random_state' : 23,
    'n_estimators' : 400,
    'num_leaves' : 36,
    'learning_rate' : 0.03,
    'subsample' : 0.9,
    'colsample_bytree' : 0.75,
    'reg_alpha' : 0.6,
    'reg_lambda' : 0.2,
    'class_weight' : {0:1, 1:10},
    'n_jobs' : -1    
}
# lgbm_best_param_org = {
#     random_state     = 23,
#     n_estimators     = 300,
#     colsample_bytree = 0.73,
#     # max_depth      = -1,
#     learning_rate    = 0.03,
#     num_leaves       = 42,
#     reg_alpha        = 0.61,
#     reg_lambda       = 0.13,
#     subsample        = 0.97,
#     class_weight     = {0:1, 1:10},
#     n_jobs           = -1,    
# }
lgbm_clf = LGBMClassifier(**lgbm_best_param)


# 2. Meta model 정의 - 윤지훈님 best param.
meta_model = LogisticRegression(
    random_state = 23,
    max_iter     = 1000,
    C            = 0.029,
    penalty      = 'l2',
    solver       = 'lbfgs',
    class_weight ="balanced"
)

# 3. StackingClassifier 구성
stacking_model = StackingClassifier(
    estimators      = [('rf', rf_clf), ('xgb', xgb_clf), ('lgbm', lgbm_clf)],
    final_estimator = meta_model,
    cv              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs          = -1
)

# 4. 학습 (예시: 전처리된 데이터 X_reduced, y_labels 사용)
stacking_model.fit(X_reduced, y_labels)


,estimators,"[('rf', ...), ('xgb', ...), ...]"
,final_estimator,LogisticRegre...ndom_state=23)
,cv,StratifiedKFo... shuffle=True)
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,390
,criterion,'gini'
,max_depth,25
,min_samples_split,7


In [27]:
# 5. 메타 모델 계수 출력
coef = stacking_model.final_estimator_.coef_[0]
base_models = ['RandomForest', 'XGBoost', 'LightGBM']
for name, weight in zip(base_models, coef):
    print(f"{name} 기여도(계수): {weight:.4f}")

RandomForest 기여도(계수): 1.8603
XGBoost 기여도(계수): -0.6759
LightGBM 기여도(계수): 4.8092


In [28]:
# 5-2. 스택킹 모델 성능 평가 (전처리된 데이터 X_reduced, y_labels 사용)
from sklearn.metrics import roc_auc_score, f1_score, recall_score

y_pred_proba = stacking_model.predict_proba(X_reduced)[:, 1]
y_pred = stacking_model.predict(X_reduced)

auc_score = roc_auc_score(y_labels, y_pred_proba)
f1 = f1_score(y_labels, y_pred)
recall = recall_score(y_labels, y_pred)

print(f"\nStacking 모델 ROC-AUC: {auc_score:.4f}")
print(f"Stacking 모델 F1-Score: {f1:.4f}")
print(f"Stacking 모델 Recall:   {recall:.4f}")


Stacking 모델 ROC-AUC: 0.9300
Stacking 모델 F1-Score: 0.2933
Stacking 모델 Recall:   0.8853


In [ ]:
#threshold  dropped_features  AUC       F1        Recall
# 0.95      53                0.842169  0.016287  0.008306 (Not Scaled)
# 0.95      53                0.8409    0.0131    0.0066   (Scaled)
# 0.95      57                0.8359    0.0350    0.0183   (Not Scaled)
# 0.95      53                0.8318    0.0218    0.0113   (KFold)

In [29]:
from utils.model_utils import save_model

save_model(stacking_model, 'StackingModel_log1p_RF+LGBM-HP+XGB+LR_20251124')

✓ 모델 저장 완료: ../models\StackingModel_log1p_RF+LGBM-HP+XGB+LR_20251124.pkl
  파일 크기: 96.94 MB


'../models\\StackingModel_log1p_RF+LGBM-HP+XGB+LR_20251124.pkl'

In [30]:
X_test_reduced.shape


(75818, 96)

In [34]:
# TEST data 예측
test_pred_proba = stacking_model.predict_proba(X_test_reduced)[:, 1]
test_pred = stacking_model.predict(X_test_reduced)

In [35]:
# Test 결과 저장 
def load_test_data(test_path="../data/test.csv"):
    test = pd.read_csv(test_path)
    return test

test = load_test_data()
submission = pd.DataFrame({'ID': test['ID'], 'TARGET': test_pred})
submission.to_csv('../results/submission_stack2.txt', index=False)

In [36]:
# 개수
print(submission['TARGET'].value_counts())

# 비율
print(submission['TARGET'].value_counts(normalize=True))

results_text = ''' 
0.9000 일때
TARGET
0    60954
1    14864
Name: count, dtype: int64
TARGET
0    0.803952
1    0.196048
Name: proportion, dtype: float64
'''


TARGET
0    60919
1    14899
Name: count, dtype: int64
TARGET
0    0.80349
1    0.19651
Name: proportion, dtype: float64
